# 🎧 Audiobook Generation System — Colab Training (H100 AUTOPILOT, resume-safe)

Trains the **ByT5 Text-Normalization** model, evaluates it against the rule baseline,
runs the **agent** on a sample book (real SpeechT5), and generates `report.pdf` + `slides.pptx`.

**Auto-adapts** to H100 / A100 / L4 / T4 (batch + precision; effective batch held constant) and
**resumes** from the last checkpoint on Drive after a disconnect. Just set the Controls and
`Runtime → Run all`.


In [ ]:
#@title 0) Controls — set these, then `Runtime → Run all`  { display-mode: "form" }
GIT_REPO_URL = "https://github.com/ledinhminhquan/05_Audiobook_Generation" #@param {type:"string"}
BRANCH = "main" #@param {type:"string"}
PROJECT_DIR_NAME = "05_Audiobook_Generation" #@param {type:"string"}
USE_DRIVE = True #@param {type:"boolean"}
DRIVE_SUBDIR = "audiobook_ai" #@param {type:"string"}
BASE_MODEL = "auto" #@param ["auto", "google/byt5-small", "google-t5/t5-small"]
TRAIN_SIZE = 60000 #@param {type:"integer"}
EPOCHS = 3 #@param {type:"integer"}
RUN_TUNE = False #@param {type:"boolean"}
TTS_BACKEND = "speecht5" #@param ["speecht5", "kokoro", "auto", "placeholder"]
print("Controls set. Repo:", GIT_REPO_URL or "(will copy from Drive)")

In [ ]:
#@title 1) Check the GPU
!nvidia-smi -L || echo "No GPU — select Runtime > Change runtime type > GPU"

In [ ]:
#@title 2) Mount Drive + artifact paths & HF caches  (BEFORE importing torch)
import os
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    ART = f"/content/drive/MyDrive/{DRIVE_SUBDIR}"
else:
    ART = "/content/audiobook_ai"
os.makedirs(ART, exist_ok=True)
os.environ["AUDIOBOOK_AI_ARTIFACTS_DIR"] = ART
os.environ["AUDIOBOOK_AI_MODEL_DIR"]     = f"{ART}/models"
os.environ["AUDIOBOOK_AI_DATA_DIR"]      = f"{ART}/data"
os.environ["AUDIOBOOK_AI_RUN_DIR"]       = f"{ART}/runs"
os.environ["AUDIOBOOK_AI_OUTPUT_DIR"]    = f"{ART}/outputs"
os.environ["HF_HOME"]                    = f"{ART}/hf_cache"
print("Artifacts (Drive-persisted) ->", ART)

In [ ]:
#@title 3) System tools (ffmpeg = mp3/m4b export, espeak-ng = pyttsx3 fallback)
!apt-get -qq install -y ffmpeg espeak-ng > /dev/null 2>&1 && echo "ffmpeg + espeak-ng ready"

In [ ]:
#@title 4) Get the project source (git clone, or copy from Drive)
import os
WORK = "/content/" + PROJECT_DIR_NAME
if GIT_REPO_URL:
    if not os.path.exists(WORK):
        rc = os.system(f'git clone -b "{BRANCH}" "{GIT_REPO_URL}" "{WORK}"')
        if rc != 0:
            os.system(f'git clone "{GIT_REPO_URL}" "{WORK}"')
else:
    src = f"/content/drive/MyDrive/{PROJECT_DIR_NAME}"
    assert os.path.exists(src), f"Set GIT_REPO_URL or upload the project to Drive/{PROJECT_DIR_NAME}"
    os.system(f'cp -r "{src}" "{WORK}"')
os.chdir(WORK)
print("cwd:", os.getcwd())

In [ ]:
#@title 5) Install dependencies (Colab-safe: NEVER reinstall torch)
%cd $WORK
!pip -q install -r requirements_colab.txt
!pip -q install -e . --no-deps
print("Dependencies installed (torch untouched).")

In [ ]:
#@title 6) Verify environment + performance knobs (TF32)
import torch, transformers
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
GPU = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
print("torch", torch.__version__, "| transformers", transformers.__version__)
print("GPU:", GPU, "| CUDA:", torch.cuda.is_available())

In [ ]:
#@title 7) Auto GPU profile (batch + precision; effective batch held at 256)
import torch
GPU = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
EFF = 256
prof = {"base_model": "google/byt5-small"}
if "H100" in GPU:        bs, bf16, fp16, ckpt = 32, True,  False, False
elif "A100" in GPU:      bs, bf16, fp16, ckpt = 16, True,  False, True
elif "L4" in GPU:        bs, bf16, fp16, ckpt = 8,  True,  False, True
elif "T4" in GPU:        bs, bf16, fp16, ckpt = 4,  False, True,  True; prof["base_model"] = "google-t5/t5-small"
else:                    bs, bf16, fp16, ckpt = 2,  False, False, True; prof["base_model"] = "google-t5/t5-small"
if BASE_MODEL != "auto":
    prof["base_model"] = BASE_MODEL
accum = max(1, EFF // bs)
prof.update(per_device_train_batch_size=bs, gradient_accumulation_steps=accum,
            bf16=bf16, fp16=fp16, gradient_checkpointing=ckpt)
print(GPU, "->", prof)

In [ ]:
#@title 8) Write the Colab training config  (configs/train_colab.yaml)
import yaml
lr = 5.0e-4 if "byt5" in prof["base_model"] else 3.0e-4
cfg = {
    "data": {"synthetic_train_size": int(TRAIN_SIZE), "synthetic_val_size": 4000,
             "synthetic_test_size": 4000, "hard_slice_size": 1500, "seed": 42},
    "model": {**prof, "num_train_epochs": int(EPOCHS), "learning_rate": lr,
              "per_device_eval_batch_size": 64, "tf32": True, "group_by_length": True,
              "label_smoothing_factor": 0.1, "early_stopping_patience": 4,
              "eval_steps": 500, "save_steps": 500},
}
open("configs/train_colab.yaml", "w").write(yaml.safe_dump(cfg, sort_keys=False))
print(open("configs/train_colab.yaml").read())

In [ ]:
#@title 9) Build + sanity-check the synthetic TN corpus
!audiobook-ai --config configs/train_colab.yaml data --task corpus

In [ ]:
#@title 10) (Optional) basic LR hyperparameter search
if RUN_TUNE:
    !audiobook-ai --config configs/train_colab.yaml tune --n-trials 3 --limit 4000
else:
    print("RUN_TUNE is off — skipping.")

## ⭐ ONE BUTTON — autopilot (resume-safe)
Trains → evaluates (vs baseline) → analysis → `report.pdf` + `slides.pptx` + bundle.
**Re-run this cell after a disconnect to RESUME** from the last checkpoint on Drive.

In [ ]:
#@title 11) ⭐ ONE BUTTON autopilot  (re-run to resume)
!audiobook-ai --config configs/train_colab.yaml autopilot

## Individual steps (optional) — idempotent + resume-safe

In [ ]:
#@title 12a) Train only (resumes from the last checkpoint)
!audiobook-ai --config configs/train_colab.yaml train

In [ ]:
#@title 12b) Evaluate — neural vs baseline, overall + the hard ambiguous slice
!audiobook-ai evaluate --which test
!audiobook-ai evaluate --which hard

In [ ]:
#@title 13) Diagnostics: eval metrics + trained-model metadata
import json, os
rd = os.environ["AUDIOBOOK_AI_RUN_DIR"]
try:
    ev = json.load(open(f"{rd}/eval/latest.json"))
    print("SUMMARY:", json.dumps(ev.get("summary", {}), indent=2))
    if ev.get("neural"):
        print("delta:", ev.get("delta"))
except Exception as e:
    print("run evaluate first:", e)

## ✅ Test the trained model

In [ ]:
#@title 14a) Normalization: trained model vs rule baseline (tricky inputs)
from audiobook_ai.agent.narrator_agent import NarratorAgent
from audiobook_ai.models.baseline_rules import normalize as rule_norm
agent = NarratorAgent(load_model=True)        # loads the fine-tuned ByT5 from Drive
tests = [
    "He paid $5.2M in 1984 and read 1984 again.",
    "Dr. Vance lives on Oak Dr. near St. Mary on Baker St.",
    "Chapter IV begins at 9:45 AM on the 21st; only 3/4 stayed.",
    "1984 people attended; call 555-203-1984 or visit www.example.com.",
]
for t in tests:
    job = agent.normalize_preview(t)
    print("IN   :", t)
    print("RULE :", rule_norm(t))
    print("MODEL:", " ".join(s.normalized for s in job.segments), "\n")

In [ ]:
#@title 14b) Make an audiobook from the sample book (real SpeechT5) + play it
import os
from IPython.display import Audio, display
from audiobook_ai.agent.narrator_agent import NarratorAgent
agent = NarratorAgent(load_model=True, tts_backend=TTS_BACKEND)
job = agent.process(path="sample_data/sample_book.txt", title="The Clockmaker's Ledger")
print("status:", job.status.value, "| metrics:", job.metrics.get("audio_duration"), "s, RTF", job.metrics.get("rtf"))
print("outputs:", list(job.outputs.keys()))
w = job.outputs.get("mp3") or job.outputs.get("wav")
if w and os.path.exists(w):
    display(Audio(w))

In [ ]:
#@title 15) Locate deliverables (report.pdf + slides.pptx + bundle)
import glob, os
sub = sorted(glob.glob(os.environ["AUDIOBOOK_AI_ARTIFACTS_DIR"] + "/submission/*"))
if sub:
    print("Submission bundle ->", sub[-1])
    for f in sorted(glob.glob(sub[-1] + "/*")):
        print("  ", os.path.basename(f), os.path.getsize(f), "bytes")
else:
    print("Run the autopilot cell (11) to produce report.pdf + slides.pptx.")

In [ ]:
#@title 16) (Optional) Serve the API + Gradio UI (share link)
# !audiobook-ai serve --ui --port 7860
print("Uncomment the line above to launch the FastAPI + Gradio app.")

## ✅ Final checklist
- [ ] GPU profile picked (cell 7) — H100/A100 → bf16, T4 → fp16 + t5-small
- [ ] Corpus built (cell 9), training finished (cell 11/12a)
- [ ] `evaluate` shows the model **beats the baseline** (cell 12b/13), especially on the **hard** slice
- [ ] Trained model normalizes the tricky inputs correctly (cell 14a)
- [ ] Audiobook + `report.pdf` + `slides.pptx` produced (cells 14b, 15)
- [ ] Everything persisted to Drive (survives disconnects; re-run cell 11 to resume)
